# Agente Speech-to-Speech open source con Qwen2.5-Omni-3B

Este notebook implementa una primera prueba reproducible de **Speech-to-Speech nativo** con `Qwen/Qwen2.5-Omni-3B`. El modelo recibe un archivo de audio y devuelve una respuesta textual y hablada:

```text
WAV del usuario -> Qwen2.5-Omni-3B -> texto + audio WAV
```

La ejecución es local y no usa OpenAI, ElevenLabs, LiveKit cloud ni ninguna API de pago. La descarga de los pesos y el consumo de CPU/GPU son recursos propios. El modelo se publica en Hugging Face bajo la licencia indicada en su model card; comprueba siempre la licencia exacta de la revisión que utilices.

> Esta versión trabaja por turnos con WAV. El micrófono en tiempo real, VAD, streaming full-duplex e interrupciones quedan para una fase posterior.

## 1. Dependencias

Ejecuta esta celda solo si faltan paquetes. En local se recomienda instalar desde PowerShell con el entorno `voiceagent` activado. En Google Colab puede activarse `INSTALL_ON_COLAB`. La instalación no requiere claves ni servicios externos de inferencia.

In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
INSTALL_ON_COLAB = False

required_packages = {
    "torch": "torch",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "soundfile": "soundfile",
    "qwen_omni_utils": "qwen-omni-utils",
}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]

if missing_packages and IN_COLAB and INSTALL_ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)
    print("Dependencias instaladas. Reinicia el kernel y ejecuta de nuevo desde la primera celda.")
elif missing_packages:
    print("Faltan dependencias:", ", ".join(missing_packages))
    print("Instala esos paquetes en el entorno voiceagent y vuelve a ejecutar esta celda.")
else:
    print("Dependencias principales disponibles.")

In [ ]:
import platform
import sys

print(f"Python: {sys.version.split()[0]}")
print(f"Ejecutable: {sys.executable}")
print(f"Sistema: {platform.platform()}")
print(f"Entorno: {'Google Colab' if IN_COLAB else 'local'}")

if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch no esta instalado en el entorno actual.")

import torch

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"
if CUDA_AVAILABLE:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"CUDA: disponible | GPU: {GPU_NAME} | VRAM: {GPU_MEMORY_GB:.1f} GB")
else:
    GPU_NAME = "CPU"
    GPU_MEMORY_GB = 0.0
    print("CUDA: no disponible. La inferencia de Qwen2.5-Omni-3B puede ser inviable o muy lenta en CPU.")

DTYPE = torch.bfloat16 if CUDA_AVAILABLE and torch.cuda.is_bf16_supported() else (torch.float16 if CUDA_AVAILABLE else torch.float32)
print(f"Dispositivo seleccionado: {DEVICE}")
print(f"Tipo numerico: {DTYPE}")

## 2. Configuracion del experimento

La ruta de entrada apunta inicialmente a `input.wav` junto al notebook. Usa un WAV corto, claro y en español. No se sube el audio a ningún servicio externo.

In [ ]:
import time
from pathlib import Path

MODEL_NAME = "Qwen/Qwen2.5-Omni-3B"
MODEL_REVISION = "main"
LANGUAGE = "es"
SYSTEM_PROMPT = "Responde en español de forma breve, clara y natural."
USER_PROMPT = "Escucha el audio del usuario y responde a su pregunta en español."
MAX_NEW_TOKENS = 256
USE_AUDIO_IN_VIDEO = False
WORKSPACE_DIR = Path.cwd()
INPUT_AUDIO = WORKSPACE_DIR / "input.wav"
OUTPUT_AUDIO = WORKSPACE_DIR / "output_qwen_omni.wav"

print(f"Modelo: {MODEL_NAME}")
print(f"Revision: {MODEL_REVISION}")
print(f"Entrada: {INPUT_AUDIO.resolve()}")
print(f"Salida: {OUTPUT_AUDIO.resolve()}")

In [ ]:
import soundfile as sf

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No se ha seleccionado ningun archivo.")
    uploaded_name = next(iter(uploaded))
    if Path(uploaded_name).suffix.lower() not in {".wav", ".flac", ".ogg"}:
        raise ValueError("Selecciona un archivo WAV, FLAC u OGG.")
    INPUT_AUDIO = Path(uploaded_name)

if not INPUT_AUDIO.exists():
    raise FileNotFoundError(f"No existe INPUT_AUDIO: {INPUT_AUDIO.resolve()}")

audio_info = sf.info(INPUT_AUDIO)
if audio_info.duration <= 0 or audio_info.duration > 30:
    raise ValueError("Usa un audio con una duracion entre 0 y 30 segundos.")

print(f"Formato: {audio_info.format}")
print(f"Frecuencia: {audio_info.samplerate} Hz")
print(f"Canales: {audio_info.channels}")
print(f"Duracion: {audio_info.duration:.2f} s")

if audio_info.channels > 1:
    print("Aviso: la utilidad oficial procesara el audio multicanal segun la API instalada.")

In [ ]:
from importlib.metadata import version

try:
    from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
    from qwen_omni_utils import process_mm_info
except ImportError as error:
    raise ImportError(
        "La API oficial de Qwen2.5-Omni no esta disponible. "
        "Instala una version reciente de transformers y qwen-omni-utils."
    ) from error

print(f"Transformers: {version('transformers')}")
print("API Qwen2.5-Omni: imports correctos")

load_started = time.perf_counter()
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype=DTYPE,
    device_map="auto" if CUDA_AVAILABLE else None,
)
if not CUDA_AVAILABLE:
    model = model.to(DEVICE)
model.eval()
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
load_seconds = time.perf_counter() - load_started
print(f"Modelo cargado en {load_seconds:.2f} s")

## 3. Inferencia audio-audio

La conversación contiene el audio del usuario y una instrucción textual. `return_audio=True` solicita al modelo tanto la respuesta textual como la hablada. La llamada sigue la interfaz oficial de Transformers para Qwen2.5-Omni.

In [ ]:
conversation = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT}],
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": str(INPUT_AUDIO)},
            {"type": "text", "text": USER_PROMPT},
        ],
    },
]

prompt_text = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=False,
)
audios, images, videos = process_mm_info(
    conversation,
    use_audio_in_video=USE_AUDIO_IN_VIDEO,
)
inputs = processor(
    text=prompt_text,
    audio=audios,
    images=images,
    videos=videos,
    return_tensors="pt",
    padding=True,
    use_audio_in_video=USE_AUDIO_IN_VIDEO,
)
inputs = {name: value.to(model.device) if hasattr(value, "to") else value for name, value in inputs.items()}

inference_started = time.perf_counter()
with torch.inference_mode():
    text_ids, audio_values = model.generate(
        **inputs,
        use_audio_in_video=USE_AUDIO_IN_VIDEO,
        return_audio=True,
        max_new_tokens=MAX_NEW_TOKENS,
    )
inference_seconds = time.perf_counter() - inference_started

response_text = processor.batch_decode(
    text_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]
print("Respuesta textual:")
print(response_text.strip())
print(f"Tiempo de inferencia: {inference_seconds:.2f} s")

In [ ]:
from IPython.display import Audio, display

if audio_values is None:
    raise RuntimeError("El modelo no devolvio audio. Comprueba return_audio=True y la version de Transformers.")

if isinstance(audio_values, (list, tuple)):
    audio_values = audio_values[0]
if hasattr(audio_values, "detach"):
    audio_values = audio_values.detach().float().cpu().numpy()
if audio_values.ndim > 1:
    audio_values = audio_values.squeeze()

output_sample_rate = getattr(model, "sampling_rate", None) or getattr(processor, "sampling_rate", None) or 24000
sf.write(OUTPUT_AUDIO, audio_values, output_sample_rate)
print(f"Audio guardado: {OUTPUT_AUDIO.resolve()}")
print(f"Frecuencia de salida: {output_sample_rate} Hz")
display(Audio(filename=str(OUTPUT_AUDIO)))

In [ ]:
result = {
    "model": MODEL_NAME,
    "revision": MODEL_REVISION,
    "language": LANGUAGE,
    "device": DEVICE,
    "dtype": str(DTYPE),
    "gpu": GPU_NAME,
    "input_duration_seconds": audio_info.duration,
    "load_seconds": load_seconds,
    "inference_seconds": inference_seconds,
    "output_duration_seconds": len(audio_values) / output_sample_rate,
}

if CUDA_AVAILABLE:
    result["max_cuda_memory_gb"] = torch.cuda.max_memory_allocated() / 1024**3

result

## 4. Pruebas y límites

Repite la inferencia con tres audios en español: una pregunta general, una frase con nombres o números y una petición ambigua. Conserva la misma configuración y registra `result` para comparar latencia, estabilidad y calidad.

Este notebook demuestra S2S nativo por turnos, pero todavía no es un agente realtime completo. No mide VAD, latencia hasta el primer fragmento de audio, interrupciones, streaming full-duplex ni conversación mediante micrófono. Esas funciones se implementarán después de validar la ruta WAV.

**Fuentes**

- Repositorio del modelo: https://github.com/QwenLM/Qwen2.5-Omni
- Model card: https://huggingface.co/Qwen/Qwen2.5-Omni-3B
- API: Transformers y `qwen-omni-utils` según las versiones instaladas

La licencia, la revisión del modelo y las licencias de dependencias deben verificarse antes de incluir resultados definitivos en el TFM.